# ARS Notebook 6 - Paper Figures and Result Archive

Create presentation-ready figures and tables from saved JSON/JSONL outputs. This notebook is CPU-only and does not load Phi-2 or repeat inference. Figures use a black background with white and orange marks to match the project deck.


## Input contract

Attach the six completed evaluation ZIPs if available: the 0-shot and 2-shot bundles for GSM8K, GSM-Plus, and ARC-Challenge. The notebook detects bundles from their contents, so the detailed filenames are accepted. It also accepts extracted result folders. Do not attach training checkpoints; this notebook is CPU-only and does not load Phi-2 or rerun inference.

Each output record should contain all four arms: baseline, steer_only, static_bank, and router. Missing dataset/shot conditions are reported as missing rather than filled in.

In [ ]:
import json, os, shutil, subprocess, sys, zipfile
from pathlib import Path, PurePosixPath

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["MPLBACKEND"] = "Agg"  # Headless PNG export on Kaggle and local kernels.
REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"
REPO_DIR = Path("/kaggle/working/adaptive-residual-steering")
WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "development"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "development"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "development"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "development", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))


def _is_evaluation_archive(path):
    try:
        with zipfile.ZipFile(path) as archive:
            members = [PurePosixPath(name) for name in archive.namelist()]
        has_analysis = any(member.name == "benchmark_metrics_summary.json" and "analysis" in member.parts for member in members)
        has_record = any(
            member.suffix == ".json"
            and member.name.startswith(("session3_", "mcq_"))
            and not any(token in member.name for token in ("status", "static_bank_action"))
            for member in members
        )
        return has_analysis and has_record
    except (OSError, zipfile.BadZipFile):
        return False


evaluation_archives = sorted(path for path in INPUT_ROOT.rglob("*.zip") if _is_evaluation_archive(path))
repo_helpers = [REPO_DIR / "scripts" / name for name in ("research_notebook_utils.py", "research_notebook_analysis.py")]
code_bootstrap = {"source": "repository", "paths": [str(path) for path in repo_helpers]} if all(path.is_file() for path in repo_helpers) else None

if code_bootstrap is None:
    for archive_path in evaluation_archives:
        with zipfile.ZipFile(archive_path) as archive:
            members = set(archive.namelist())
            required = {"code/scripts/research_notebook_utils.py", "code/scripts/research_notebook_analysis.py"}
            if required.issubset(members):
                for name in ("research_notebook_utils.py", "research_notebook_analysis.py"):
                    (REPO_DIR / "scripts" / name).write_bytes(archive.read(f"code/scripts/{name}"))
                code_bootstrap = {"source": "evaluation_archive_fallback", "path": str(archive_path)}
                break

if code_bootstrap is None:
    for analysis_path in sorted(INPUT_ROOT.rglob("research_notebook_analysis.py")):
        utils_path = analysis_path.with_name("research_notebook_utils.py")
        if analysis_path.parent.name == "scripts" and utils_path.is_file():
            for source, name in ((utils_path, "research_notebook_utils.py"), (analysis_path, "research_notebook_analysis.py")):
                shutil.copy2(source, REPO_DIR / "scripts" / name)
            code_bootstrap = {"source": "extracted_evaluation_files", "path": str(analysis_path.parent)}
            break

if code_bootstrap is None:
    raise FileNotFoundError("Analysis helpers are missing from the repository and attached evaluation files.")

from scripts.research_notebook_analysis import collect_saved_records, create_paper_assets
from scripts.research_notebook_utils import merge_result_bundle, sha256_file

RESULTS_ROOT = WORK_ROOT / "ars_all_saved_results"
if RESULTS_ROOT.exists():
    shutil.rmtree(RESULTS_ROOT)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
loaded_bundles = []
source_mode = None

if evaluation_archives:
    source_mode = "evaluation_zip_archives"
    seen_archives = set()
    for archive_path in evaluation_archives:
        archive_hash = sha256_file(archive_path)
        if archive_hash in seen_archives:
            continue
        seen_archives.add(archive_hash)
        bundle_dir = RESULTS_ROOT / "bundles" / f"{archive_path.stem}_{archive_hash[:12]}"
        merge_result_bundle(archive_path, bundle_dir)
        loaded_bundles.append({"name": archive_path.name, "path": str(archive_path), "sha256": archive_hash})
else:
    source_mode = "extracted_evaluation_folders"
    seen_summaries = set()
    for summary_path in sorted(INPUT_ROOT.rglob("benchmark_metrics_summary.json")):
        run_root = summary_path.parent.parent
        if not any(run_root.glob("session3_*.json")) and not any(run_root.glob("mcq_*.json")):
            continue
        summary_hash = sha256_file(summary_path)
        if summary_hash in seen_summaries:
            continue
        seen_summaries.add(summary_hash)
        bundle_dir = RESULTS_ROOT / "bundles" / f"{run_root.name}_{summary_hash[:12]}"
        shutil.copytree(run_root, bundle_dir, dirs_exist_ok=True)
        loaded_bundles.append({"name": run_root.name, "path": str(run_root), "summary_sha256": summary_hash})

if not loaded_bundles:
    raise FileNotFoundError("No evaluation result ZIPs or extracted result folders were found. Attach the completed GSM8K, GSM-Plus, and ARC-Challenge result bundles.")

saved_records = collect_saved_records([RESULTS_ROOT])
if not saved_records:
    raise ValueError("Evaluation inputs were found, but no saved four-arm benchmark records could be read.")
INPUT_REPORT = {
    "input_mode": source_mode,
    "analysis_code": code_bootstrap or {"source": "repository"},
    "bundle_count": len(loaded_bundles),
    "record_count": len(saved_records),
    "coverage": [
        {
            "dataset": record.get("suite", record.get("dataset")),
            "shots": sorted(record.get("shots", {}).keys(), key=str),
            "four_arm_complete": bool(record.get("four_arm_complete", False)),
        }
        for _, record in saved_records
    ],
    "bundles": loaded_bundles,
}
print(json.dumps(INPUT_REPORT, indent=2, ensure_ascii=False), flush=True)

## Generate assets

The saved accuracy for GSM8K/GSM-Plus is lenient numeric accuracy; ARC-Challenge uses exact choice accuracy. CSV tables retain strict, marker, and lenient scores where available. The figures include accuracy, paired intervals, wins/losses/ties, transfer, route and alpha use, completion length, latency/throughput, and per-question benefit/harm. Route diversity is descriptive and should be interpreted with paired score changes.

In [ ]:
ASSET_DIR = WORK_ROOT / "ars_paper_assets"
assets = create_paper_assets([RESULTS_ROOT], ASSET_DIR)
contact_sheet = ASSET_DIR / "figures" / "00_figure_contact_sheet.png"
if contact_sheet.is_file():
    from IPython.display import Image, display
    display(Image(filename=str(contact_sheet)))
print({"figures": assets["figures"], "figure_count": assets["figure_count"],
       "summary": assets["summary"], "bundle": assets["bundle"]})

## Export

Download `ars_paper_assets.zip` from `/kaggle/working/ars_paper_assets/`. It contains the PNG figures (including a contact sheet), CSV tables, JSON summary, and a manifest listing source records and figure names. No training bundle is needed.